In [ ]:
# dataset github link:
# 
# https://github.com/ad-freiburg/elevant/blob/master/benchmarks/aida-conll-dev.benchmark.jsonl

# this dataset was selected as it was not limited to only named entites.

# research paper link: https://aclanthology.org/2022.emnlp-demos.8.pdf

In [30]:

import json
import pandas as pd

rows=[json.loads(l) for l in open('aida-conll-dev.benchmark.jsonl')]

df = pd.DataFrame(rows)


# Flatten: one row per entity mention
flat_rows = []
for _, row in df.iterrows():
    for lab in row["labels"]:
        start, end = lab["span"]
        flat_rows.append({
            "document_id": row["id"],
            "text": row["text"],  # raw document text (the mention's context)
            "mention": row["text"][start:end],
            "start": start,
            "end": end,
            "entity_id": lab["entity_id"],
            "entity_name": lab["name"],
            "entity_types": lab["type"],
        })

flat_df = pd.DataFrame(flat_rows)
print(flat_df.shape)
flat_df.head(10)

set(flat_df["entity_types"])  # unique entity types in the dataset


(5917, 8)


{'OTHER',
 'Q12136|Q169872',
 'Q1656682',
 'Q1656682|Q2424752|Q618779',
 'Q1656682|Q27096213|Q43229',
 'Q1656682|Q349',
 'Q1656682|Q349|Q43229',
 'Q1656682|Q43229',
 'Q17376908',
 'Q17537576|Q2424752',
 'Q17537576|Q2424752|Q43229',
 'Q215627',
 'Q215627|Q43229',
 'Q2424752',
 'Q2424752|Q27096213',
 'Q2424752|Q27096213|Q4164871|Q43229|Q618779',
 'Q2424752|Q431289|Q43229',
 'Q2424752|Q43229',
 'Q27096213',
 'Q27096213|Q4164871|Q43229',
 'Q27096213|Q43229',
 'Q27096213|Q43229|Q7257',
 'Q33829',
 'Q33829|Q4392985',
 'Q4164871',
 'Q43229',
 'Q7257',
 'Q7257|Q9174'}

In [31]:
flat_df

,document_id,text,mention,start,end,entity_id,entity_name,entity_types
0,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LEICESTERSHIRE,10,24,Q3229147,Leicestershire County Cricket Club,Q43229
1,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LONDON,66,72,Q84,London,Q27096213|Q43229
2,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,West Indian,84,95,Q912881,West Indies cricket team,Q43229
3,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Phil Simmons,108,120,Q6827914,Phil Simmons,Q215627
4,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Leicestershire,151,165,Q3229147,Leicestershire County Cricket Club,Q43229
...,...,...,...,...,...,...,...,...
5912,1161,Dhaka ****** seen steady in absence of big pla...,Bangladesh Lamps,634,650,<NIL>,Unknown,OTHER
5913,1161,Dhaka ****** seen steady in absence of big pla...,Chittagong Cement,653,670,<NIL>,Unknown,OTHER
5914,1161,Dhaka ****** seen steady in absence of big pla...,Atlas Bangladesh,675,691,<NIL>,Unknown,OTHER
5915,1161,Dhaka ****** seen steady in absence of big pla...,DSE,853,856,<NIL>,Unknown,OTHER


### validating whether the given QIDs still exists or not

In [32]:
import re
import time
from pathlib import Path

import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
CACHE_FILE = Path("qid_status_cache.json")

session = requests.Session()
session.headers["User-Agent"] = "qid-validator/0.1 (educational; itisha)"


def _fetch_batch(batch: list[str], max_retries: int = 5) -> dict:
    """One API call with retry/backoff on 429."""
    for attempt in range(max_retries):
        resp = session.get(WIKIDATA_API, params={
            "action": "query",
            "titles": "|".join(batch),
            "prop": "info|pageterms",
            "inprop": "url",
            "wbptterms": "label",
            "redirects": 1,
            "format": "json",
        }, timeout=30)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt * 5))
            print(f"rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()["query"]
    raise RuntimeError(f"still rate-limited after {max_retries} retries")


def check_qids(qids: list[str]) -> dict[str, dict]:
    # load cache: QIDs checked in a previous run are never re-queried
    cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

    unique = list(dict.fromkeys(str(q) for q in qids))
    # sentinels like <NIL>, <NO_MAPPING>, OTHER are not Wikidata items
    queryable = [q for q in unique if re.fullmatch(r"Q\d+", q)]
    todo = [q for q in queryable if q not in cache]
    print(f"{len(unique)} unique ids, {len(queryable)} queryable, {len(todo)} not yet cached")

    for i in range(0, len(todo), 50):
        batch = todo[i : i + 50]
        data = _fetch_batch(batch)

        redirect_map = {r["from"]: r["to"] for r in data.get("redirects", [])}
        pages = {p["title"]: p for p in data.get("pages", {}).values()}

        for qid in batch:
            final_title = redirect_map.get(qid, qid)
            page = pages.get(final_title, {})
            if not page or "missing" in page:
                cache[qid] = {"valid": False, "label": None,
                              "redirects_to": None, "url": None}
            else:
                labels = page.get("terms", {}).get("label", [])
                cache[qid] = {
                    "valid": True,
                    "label": labels[0] if labels else None,
                    "redirects_to": final_title if final_title != qid else None,
                    "url": page["canonicalurl"],
                }

        # save progress after every batch, so an interruption loses nothing
        CACHE_FILE.write_text(json.dumps(cache))
        time.sleep(1.5)  # stay under the rate limit

    return {q: cache[q] for q in queryable}

In [33]:
import json
import re
import time
from pathlib import Path

import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
CACHE_FILE = Path("qid_status_cache.json")

session = requests.Session()
session.headers["User-Agent"] = "qid-validator/0.1 (educational; itisha)"


def _fetch_batch(batch: list[str], max_retries: int = 5) -> dict:
    """One wbgetentities call with retry/backoff on 429."""
    for attempt in range(max_retries):
        resp = session.get(WIKIDATA_API, params={
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels|descriptions",
            "languages": "en",
            "languagefallback": 1,  # items whose label lives under "mul" still resolve
            "format": "json",
        }, timeout=30)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt * 5))
            print(f"rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()["entities"]
    raise RuntimeError(f"still rate-limited after {max_retries} retries")


def check_qids(qids: list[str]) -> dict[str, dict]:
    # load cache: QIDs checked in a previous run are never re-queried
    cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

    unique = list(dict.fromkeys(str(q) for q in qids))
    # sentinels like <NIL>, <NO_MAPPING>, OTHER are not Wikidata items
    queryable = [q for q in unique if re.fullmatch(r"Q\d+", q)]
    todo = [q for q in queryable if q not in cache]
    print(f"{len(unique)} unique ids, {len(queryable)} queryable, {len(todo)} not yet cached")

    for i in range(0, len(todo), 50):
        batch = todo[i : i + 50]
        entities = _fetch_batch(batch)

        # a redirected qid comes back keyed by its target id, with a redirects.from field
        redirect_from = {e.get("redirects", {}).get("from"): k
                         for k, e in entities.items() if e.get("redirects")}
        for qid in batch:
            final = redirect_from.get(qid, qid)
            ent = entities.get(qid) or entities.get(final)
            if not ent or "missing" in ent:
                cache[qid] = {"valid": False, "label": None, "description": None,
                              "redirects_to": None, "url": None}
            else:
                cache[qid] = {
                    "valid": True,
                    "label": ent.get("labels", {}).get("en", {}).get("value"),
                    "description": ent.get("descriptions", {}).get("en", {}).get("value"),
                    "redirects_to": final if final != qid else None,
                    "url": f"https://www.wikidata.org/wiki/{ent.get('id', qid)}",
                }

        # save progress after every batch, so an interruption loses nothing
        CACHE_FILE.write_text(json.dumps(cache))
        time.sleep(1.5)  # stay under the rate limit

    return {q: cache[q] for q in queryable}

In [34]:
# validate the gold entity QIDs
status = check_qids(flat_df["entity_id"].tolist())

def _field(col: str):
    return flat_df["entity_id"].map(lambda q: status.get(str(q), {}).get(col))

flat_df["qid_valid"] = _field("valid").fillna(False).astype(bool)
flat_df["wd_label"]       = _field("label")
flat_df["wd_description"] = _field("description")
flat_df["wd_url"]         = _field("url")

# fetch names for the type QIDs (pipe-separated, e.g. "Q27096213|Q43229")
type_qids = sorted({t for ts in flat_df["entity_types"] for t in str(ts).split("|")})
type_status = check_qids(type_qids)
type_label = {q: s["label"] for q, s in type_status.items()}

flat_df["type_labels"] = flat_df["entity_types"].map(
    lambda ts: "|".join(type_label.get(t) or t for t in str(ts).split("|"))
)

print(flat_df.qid_valid.value_counts())
print(flat_df.loc[~flat_df.qid_valid, "entity_id"].value_counts())
flat_df.head(10)

1644 unique ids, 1642 queryable, 0 not yet cached
18 unique ids, 17 queryable, 0 not yet cached
qid_valid
True     4789
False    1128
Name: count, dtype: int64
entity_id
<NIL>           1126
<NO_MAPPING>       2
Name: count, dtype: int64


,document_id,text,mention,start,end,entity_id,entity_name,entity_types,qid_valid,wd_label,wd_description,wd_url,type_labels
0,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LEICESTERSHIRE,10,24,Q3229147,Leicestershire County Cricket Club,Q43229,True,Leicestershire County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3229147,organization
1,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LONDON,66,72,Q84,London,Q27096213|Q43229,True,London,capital and largest city of England and the Un...,https://www.wikidata.org/wiki/Q84,geographic entity|organization
2,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,West Indian,84,95,Q912881,West Indies cricket team,Q43229,True,West Indies cricket team,multinational cricket team,https://www.wikidata.org/wiki/Q912881,organization
3,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Phil Simmons,108,120,Q6827914,Phil Simmons,Q215627,True,Phil Simmons,West Indian cricketer,https://www.wikidata.org/wiki/Q6827914,person
4,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Leicestershire,151,165,Q3229147,Leicestershire County Cricket Club,Q43229,True,Leicestershire County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3229147,organization
5,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Somerset,171,179,Q3489807,Somerset County Cricket Club,Q43229,True,Somerset County Cricket Club,English county cricket club,https://www.wikidata.org/wiki/Q3489807,organization
6,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Essex,336,341,Q3058749,Essex County Cricket Club,Q43229,True,Essex County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3058749,organization
7,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Derbyshire,344,354,Q1788018,Derbyshire County Cricket Club,Q43229,True,Derbyshire County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q1788018,organization
8,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Surrey,359,365,Q2725632,Surrey County Cricket Club,Q43229,True,Surrey County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q2725632,organization
9,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Kent,397,401,Q3195284,Kent County Cricket Club,Q43229,True,Kent County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3195284,organization


In [35]:
set(flat_df["type_labels"])

{'OTHER',
 'creative work|product',
 'creative work|product|organization',
 'disease|symptom',
 'geographic entity',
 'geographic entity|organization',
 'geographic entity|organization|ideology',
 'geographic entity|position|organization',
 'human population',
 'human population|religious identity',
 'ideology',
 'ideology|religion',
 'languoid',
 'organization',
 'person',
 'person|organization',
 'planned event',
 'planned event|geographic entity|organization',
 'planned event|organization',
 'planned event|product|award',
 'planned event|sport',
 'planned event|sport|organization',
 'position',
 'product',
 'product|brand|organization',
 'product|geographic entity',
 'product|geographic entity|position|organization|award',
 'product|organization'}

In [40]:
# mentions with no gold Wikidata link (<NIL> = not in any KB, <NO_MAPPING> = no Wikidata mapping)
nil_df = flat_df[~flat_df["entity_id"].str.fullmatch(r"Q\d+")]

# the complement: mentions with a real gold QID
linked_df = flat_df[flat_df["entity_id"].str.fullmatch(r"Q\d+")]

print(f"NIL/no-qid mentions: {len(nil_df)}  |  linked mentions: {len(linked_df)}")
print(nil_df["entity_id"].value_counts())
nil_df.head(10)

NIL/no-qid mentions: 1128  |  linked mentions: 4789
entity_id
<NIL>           1126
<NO_MAPPING>       2
Name: count, dtype: int64


,document_id,text,mention,start,end,entity_id,entity_name,entity_types,qid_valid,wd_label,wd_description,wd_url,type_labels
41,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Tim O'Gorman,1963,1975,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
46,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,ex-England,2251,2261,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
56,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,P. Simmons,239,249,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
62,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,A. Dale,364,371,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
63,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,H. Morris,377,386,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
64,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,D. Blenkiron,392,404,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
66,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,S. Watkin,427,436,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
69,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,P. Johnson,495,505,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
70,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,M. McCague,511,521,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER
76,947,CRICKET - ENGLISH COUNTY CHAMPIONSHIP ****** ....,C. Lewis,600,608,<NIL>,Unknown,OTHER,False,NaN,NaN,NaN,OTHER


In [42]:
linked_df

,document_id,text,mention,start,end,entity_id,entity_name,entity_types,qid_valid,wd_label,wd_description,wd_url,type_labels
0,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LEICESTERSHIRE,10,24,Q3229147,Leicestershire County Cricket Club,Q43229,True,Leicestershire County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3229147,organization
1,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,LONDON,66,72,Q84,London,Q27096213|Q43229,True,London,capital and largest city of England and the Un...,https://www.wikidata.org/wiki/Q84,geographic entity|organization
2,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,West Indian,84,95,Q912881,West Indies cricket team,Q43229,True,West Indies cricket team,multinational cricket team,https://www.wikidata.org/wiki/Q912881,organization
3,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Phil Simmons,108,120,Q6827914,Phil Simmons,Q215627,True,Phil Simmons,West Indian cricketer,https://www.wikidata.org/wiki/Q6827914,person
4,946,CRICKET - LEICESTERSHIRE **** **** AT *** AFTE...,Leicestershire,151,165,Q3229147,Leicestershire County Cricket Club,Q43229,True,Leicestershire County Cricket Club,English cricket club,https://www.wikidata.org/wiki/Q3229147,organization
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5903,1160,Controversial IRA film ******** at Venice fest...,Ireland,2946,2953,Q22890,Ireland,Q27096213,True,Ireland,island in the North Atlantic Ocean,https://www.wikidata.org/wiki/Q22890,geographic entity
5904,1161,Dhaka ****** seen steady in absence of big pla...,Dhaka,0,5,Q1974408,Dhaka Stock Exchange,Q43229,True,Dhaka Stock Exchange,"Stock exchange located in Dhaka, Bangladesh",https://www.wikidata.org/wiki/Q1974408,organization
5905,1161,Dhaka ****** seen steady in absence of big pla...,DHAKA,53,58,Q1354,Dhaka,Q27096213|Q43229,True,Dhaka,capital and largest city of Bangladesh,https://www.wikidata.org/wiki/Q1354,geographic entity|organization
5906,1161,Dhaka ****** seen steady in absence of big pla...,Dhaka Stock Exchange,84,104,Q1974408,Dhaka Stock Exchange,Q43229,True,Dhaka Stock Exchange,"Stock exchange located in Dhaka, Bangladesh",https://www.wikidata.org/wiki/Q1974408,organization


In [43]:
linked_df.to_csv("el_test_data_2.csv", index=False)

In [44]:
linked_df.shape

(4789, 13)

### flaging ambiguous vs non-ambiguous QIDs

In [ ]:
# ambiguous mentions: same surface form linked to multiple QIDs



amb = (linked_df.assign(s=surface)
       .groupby("s")
       .agg(n_qids=("entity_id", "nunique"),
            qids=("entity_id", lambda x: sorted(set(x))),
            labels=("wd_label", lambda x: " / ".join(sorted(set(x)))))
       .query("n_qids > 1")
       .sort_values("n_qids", ascending=False))
amb

,n_qids,qids,labels
s,,,
england,5,"[Q1321565, Q179876, Q21, Q378628, Q47762]",England / England cricket team / England men's...
northampton,4,"[Q1201283, Q192240, Q3344193, Q48945]",Northampton / Northampton Saints / Northampton...
chicago,4,"[Q1297, Q246782, Q2633007, Q335169]",Chicago / Chicago Cubs / Chicago White Sox / N...
los angeles,3,"[Q334634, Q65, Q743309]",Los Angeles / Los Angeles Angels / Los Angeles...
bulgaria,3,"[Q180845, Q219, Q651714]",Bulgaria / Bulgaria men's national football te...
...,...,...,...
hove,2,"[Q5177658, Q989616]","County Cricket Ground, Hove / Hove"
harlequins,2,"[Q1149835, Q863469]",Harlequin F.C. / London Broncos
gulf,2,"[Q34675, Q37643]",Gulf War / Persian Gulf


In [62]:
RNG = 42
s = linked_df["mention"].str.lower()
linked_df = linked_df.assign(s=s)

ambig_forms = linked_df.groupby("s")["entity_id"].nunique().loc[lambda x: x > 1].index

# one randomly chosen occurrence of each ambiguous surface form (116 rows)
amb_df = (linked_df[linked_df.s.isin(ambig_forms)]
          .groupby("s", as_index=False).sample(1, random_state=RNG)
          .assign(category="ambiguous"))

# one row per non-ambiguous surface form, sampled down to the same size (116 rows)
clear_df = (linked_df[~linked_df.s.isin(ambig_forms)]
            .drop_duplicates("s")
            .sample(len(amb_df), random_state=RNG)
            .assign(category="non_ambiguous"))

eval_df = pd.concat([amb_df, clear_df], ignore_index=True).drop(columns="s")

In [64]:
eval_df.to_csv("el_stratified_dataset.csv", index=False)

In [65]:
eval_df.shape

(232, 16)